In [1]:
import pandas as pd
import glob
import os
import urllib.request

from debugpy.common.timestamp import current

folder_path = "../data/Premier_League/Not_merged"
url_live = "https://www.football-data.co.uk/mmz4281/2526/E0.csv"
live_file_path = os.path.join(folder_path, "E0_25_26_LIVE.csv")
urllib.request.urlretrieve(url_live, live_file_path)

file_pattern = os.path.join(folder_path, "*.csv")
files = glob.glob(file_pattern)

print(f"Found {len(files)} files")

Found 26 files


# Data Loading and Merging
*TODO: I need to download data not only for the 2025/2026 season, but for all seasons starting from around 2000.*

In this section, I am loading all the individual CSV files for different Premier League seasons, defining a set of core columns that I want to keep, and then merging them into a single master DataFrame.
I also handle potential encoding and date parsing errors during the file reading process.

In [2]:
core_columns =[
    'Div', 'Date', 'HomeTeam', 'AwayTeam',
    'FTHG', 'FTAG', 'FTR', 'HTHG', 'HTAG', 'HTR',
    'Referee',
    'HS', 'AS', 'HST', 'AST', 'HC', 'AC',
    'HF', 'AF', 'HY', 'AY', 'HR', 'AR'
]

all_matches = []

for file in files:
    df = pd.read_csv(file, encoding='utf-8', on_bad_lines='skip')

    if 'Date' in df.columns:
        df['Date'] = pd.to_datetime(df['Date'], dayfirst=True, errors='coerce', format='mixed')

    cols_to_keep = [col for col in core_columns if col in df.columns]
    df = df[cols_to_keep]
    all_matches.append(df)


In [3]:
master_db = pd.concat(all_matches, ignore_index=True)

# Initial Data Cleaning
Here, I'm performing some initial cleaning on the merged data. This includes dropping rows with missing essential data like dates or team names, and then sorting the data by date.

In [4]:
master_db = master_db.dropna(subset=['Date','HomeTeam', 'AwayTeam'])
master_db=master_db.sort_values(by='Date').reset_index(drop=True)

# Season Feature Engineering
I'm creating a 'Season' column based on the match date. This will be useful for season-specific analysis and for the ELO rating system later on.

In [5]:
def get_season(date):
    if pd.isna(date):
        return "Unknown"
    year = date.year
    month = date.month
    if month >= 7:
        return f"{year}/{year+1}"
    else:
        return f"{year-1}/{year}"

master_db['Season'] = master_db['Date'].apply(get_season)

In [6]:
output_dir = "../data/Premier_League"
output_filename = os.path.join(output_dir, "PremierLeague_WszystkieSezony.csv")

master_db.to_csv(output_filename, index=False)

**The Premier League data has been merged, pre-processed, and saved to a single file.**

# Target Variable and Points Calculation
I am creating the `Target` variable, which is a numerical representation of the match outcome (Home win, Draw, Away win). I am also calculating the points awarded to the home and away teams for each match.

In [7]:
df=master_db.copy()
df.isna().sum()

Div         0
Date        0
HomeTeam    0
AwayTeam    0
FTHG        0
FTAG        0
FTR         0
HTHG        0
HTAG        0
HTR         0
Referee     0
HS          0
AS          0
HST         0
AST         0
HC          0
AC          0
HF          0
AF          0
HY          0
AY          0
HR          0
AR          0
Season      0
dtype: int64

In [8]:
df = master_db.copy()
ftr_mapping = {'H': 2, 'D': 1, 'A': 0}
df['Target'] = df['FTR'].map(ftr_mapping)

In [9]:
def get_home_points(ftr):
    if ftr == 'H': return 3
    elif ftr == 'D': return 1
    else: return 0

def get_away_points(ftr):
    if ftr == 'A': return 3
    elif ftr == 'D': return 1
    else: return 0

In [10]:
df['HomePoints'] = df['FTR'].apply(get_home_points)
df['AwayPoints'] = df['FTR'].apply(get_away_points)

print(df[['Date', 'HomeTeam', 'AwayTeam','FTR', 'HomePoints', 'AwayPoints']].head(10))

        Date    HomeTeam       AwayTeam FTR  HomePoints  AwayPoints
0 2000-08-19       Leeds        Everton   H           3           0
1 2000-08-19   Liverpool       Bradford   H           3           0
2 2000-08-19  Sunderland        Arsenal   H           3           0
3 2000-08-19    Charlton       Man City   H           3           0
4 2000-08-19     Chelsea       West Ham   H           3           0
5 2000-08-19    Coventry  Middlesbrough   A           0           3
6 2000-08-19       Derby    Southampton   D           1           1
7 2000-08-19   Tottenham        Ipswich   H           3           0
8 2000-08-19   Leicester    Aston Villa   D           1           1
9 2000-08-20  Man United      Newcastle   H           3           0


# Days of Rest Calculation
To account for team fatigue, I'm calculating the number of days of rest each team had before a match. I also identify new "spells" for a team, which could indicate a long break (e.g., between seasons or due to relegation/promotion). This helps in creating a more accurate representation of a team's current state.

In [11]:
df['Date']= pd.to_datetime(df['Date'])
df = df.sort_values(by='Date').reset_index(drop=True)

df['HomeTeam'] = df['HomeTeam'].astype(str).str.split('_').str[0]
df['AwayTeam'] = df['AwayTeam'].astype(str).str.split('_').str[0]

home_dates = df[['Date', 'HomeTeam']].rename(columns={'HomeTeam':'Team'})
away_dates = df[['Date', 'AwayTeam']].rename(columns={'AwayTeam':'Team'})
all_matches = pd.concat([home_dates, away_dates]).sort_values(by=['Team','Date']).reset_index(drop=True)

all_matches['Raw_DaysOfRest']= all_matches.groupby('Team')['Date'].diff().dt.days
all_matches['Is_New_Spell'] = (all_matches['Raw_DaysOfRest'] > 300).astype(int)
all_matches['Spell_ID']= all_matches.groupby('Team')['Is_New_Spell'].cumsum()

all_matches['Team_Spell'] = all_matches['Team'] + '_' + all_matches['Spell_ID'].astype(str)

all_matches['DaysOfRest'] = all_matches['Raw_DaysOfRest'].apply(lambda x: 14.0 if pd.isna(x) or x>70 else x)



In [12]:
df = df.merge(all_matches[['Date','Team','DaysOfRest', 'Team_Spell']], left_on=['Date', 'HomeTeam'], right_on=['Date', 'Team'], how='left')
df.rename(columns={'DaysOfRest':'Home_DaysOfRest'}, inplace=True)

df['HomeTeam']=df['Team_Spell']
df.drop(['Team','Team_Spell'], axis=1, inplace=True)

df = df.merge(all_matches[['Date','Team', 'DaysOfRest', 'Team_Spell']], left_on=['Date', 'AwayTeam'], right_on=['Date', 'Team'], how='left')
df.rename(columns={'DaysOfRest':'Away_DaysOfRest'}, inplace=True)
df['AwayTeam']=df['Team_Spell']
df.drop(['Team','Team_Spell'], axis=1, inplace=True)

# Rolling Averages for Home and Away Form
I'm calculating rolling averages for various statistics (points, goals, shots, etc.) for both home and away teams based on their last 5 home and away matches respectively. This will give me an indication of their recent form in those specific conditions.

In [13]:
df=df.sort_values(by='Date').reset_index(drop=True)

#home stats
df['Home_Last5_HomePts']=df.groupby('HomeTeam')['HomePoints'].transform(lambda x: x.shift(1).rolling(5,min_periods=1).mean())

#offensive home stats
df['Home_Last5_GF']=df.groupby('HomeTeam')['FTHG'].transform(lambda x: x.shift(1).rolling(5,min_periods=1).mean())
df['Home_Last5_HomeShots']=df.groupby('HomeTeam')['HS'].transform(lambda x: x.shift(1).rolling(5,min_periods=1).mean())
df['Home_Last5_HomeShotsOT']=df.groupby('HomeTeam')['HST'].transform(lambda x: x.shift(1).rolling(5,min_periods=1).mean())
df['Home_Last5_Corners']=df.groupby('HomeTeam')['HC'].transform(lambda x: x.shift(1).rolling(5,min_periods=1).mean())
df['Home_Last5_HomeFouls']=df.groupby('HomeTeam')['HF'].transform(lambda x: x.shift(1).rolling(5,min_periods=1).mean())
df['Home_Last5_HomeYellows']=df.groupby('HomeTeam')['HY'].transform(lambda x: x.shift(1).rolling(5,min_periods=1).mean())
df['Home_Last5_HomeReds']=df.groupby('HomeTeam')['HR'].transform(lambda x: x.shift(1).rolling(5,min_periods=1).mean())

#defensive home stats
df['Home_Last5_GA']=df.groupby('HomeTeam')['FTAG'].transform(lambda x: x.shift(1).rolling(5,min_periods=1).mean())
df['Home_Last5_HomeShotsAgainst']=df.groupby('HomeTeam')['AS'].transform(lambda x:x.shift(1).rolling(5,min_periods=1).mean())
df['Home_Last5_HomeShotsOTAgainst']=df.groupby('HomeTeam')['AST'].transform(lambda x: x.shift(1).rolling(5,min_periods=1).mean())
df['Home_Last5_CornersAgainst']=df.groupby('HomeTeam')['AC'].transform(lambda x: x.shift(1).rolling(5,min_periods=1).mean())
df['Home_Last5_HomeFoulsAgainst']=df.groupby('HomeTeam')['AF'].transform(lambda x: x.shift(1).rolling(5,min_periods=1).mean())
df['Home_Last5_HomeYellowsAgainst']=df.groupby('HomeTeam')['AY'].transform(lambda x: x.shift(1).rolling(5,min_periods=1).mean())
df['Home_Last5_HomeRedsAgainst']=df.groupby('HomeTeam')['AR'].transform(lambda x: x.shift(1).rolling(5,min_periods=1).mean())

In [14]:
#away stats
df['Away_Last5_AwayPts']=df.groupby('AwayTeam')['AwayPoints'].transform(lambda x: x.shift(1).rolling(5,min_periods=1).mean())

#away offensive stats
df['Away_Last5_GF']=df.groupby('AwayTeam')['FTAG'].transform(lambda x: x.shift(1).rolling(5,min_periods=1).mean())
df['Away_Last5_AwayShots']=df.groupby('AwayTeam')['AS'].transform(lambda x: x.shift(1).rolling(5,min_periods=1).mean())
df['Away_Last5_AwayShotsOT']=df.groupby('AwayTeam')['AST'].transform(lambda x: x.shift(1).rolling(5,min_periods=1).mean())
df['Away_Last5_Corners']=df.groupby('AwayTeam')['AC'].transform(lambda x: x.shift(1).rolling(5,min_periods=1).mean())
df['Away_Last5_AwayFouls']=df.groupby('AwayTeam')['AF'].transform(lambda x: x.shift(1).rolling(5,min_periods=1).mean())
df['Away_Last5_AwayYellows']=df.groupby('AwayTeam')['AY'].transform(lambda x: x.shift(1).rolling(5,min_periods=1).mean())
df['Away_Last5_AwayReds']=df.groupby('AwayTeam')['AR'].transform(lambda x: x.shift(1).rolling(5,min_periods=1).mean())
#away defensive stats
df['Away_Last5_GA']=df.groupby('AwayTeam')['FTHG'].transform(lambda x: x.shift(1).rolling(5,min_periods=1).mean())
df['Away_Last5_AwayShotsAgainst']=df.groupby('AwayTeam')['HS'].transform(lambda x: x.shift(1).rolling(5,min_periods=1).mean())
df['Away_Last5_AwayShotsOTAgainst']=df.groupby('AwayTeam')['HST'].transform(lambda x: x.shift(1).rolling(5,min_periods=1).mean())
df['Away_Last5_CornersAgainst']=df.groupby('AwayTeam')['HC'].transform(lambda x: x.shift(1).rolling(5,min_periods=1).mean())
df['Away_Last5_AwayFoulsAgainst']=df.groupby('AwayTeam')['HF'].transform(lambda x: x.shift(1).rolling(5,min_periods=1).mean())
df['Away_Last5_AwayYellowsAgainst']=df.groupby('AwayTeam')['HY'].transform(lambda x: x.shift(1).rolling(5,min_periods=1).mean())
df['Away_Last5_AwayRedsAgainst']=df.groupby('AwayTeam')['HR'].transform(lambda x: x.shift(1).rolling(5,min_periods=1).mean())

# Overall Team Form
To get a more general sense of a team's form, I'm combining their home and away stats and then calculating rolling averages over their last 5 matches, regardless of venue.

In [15]:
home_form=df[['Date','HomeTeam','HomePoints','FTHG','FTAG','HS','AS','HST','AST','HC','AC','HF','AF','HY','AY','HR','AR']].copy()
home_form.columns = ['Date', 'Team', 'Points', 'GoalsFor', 'GoalsAgainst', 'ShotsFor', 'ShotsAgainst', 'ShotsOTFor', 'ShotsOTAgainst', 'CornersFor', 'CornersAgainst', 'FoulsFor', 'FoulsAgainst', 'YellowsFor', 'YellowsAgainst', 'RedsFor', 'RedsAgainst']

away_form = df[['Date', 'AwayTeam', 'AwayPoints', 'FTAG', 'FTHG', 'AS', 'HS', 'AST', 'HST', 'AC', 'HC', 'AF', 'HF', 'AY', 'HY', 'AR', 'HR']].copy()
away_form.columns = ['Date', 'Team', 'Points', 'GoalsFor', 'GoalsAgainst', 'ShotsFor', 'ShotsAgainst', 'ShotsOTFor', 'ShotsOTAgainst', 'CornersFor', 'CornersAgainst', 'FoulsFor', 'FoulsAgainst', 'YellowsFor', 'YellowsAgainst', 'RedsFor', 'RedsAgainst']


In [16]:
team_form=pd.concat([home_form, away_form]).sort_values(by=['Team','Date']).reset_index(drop=True)
stat_columns = ['Points', 'GoalsFor', 'GoalsAgainst', 'ShotsFor', 'ShotsAgainst', 'ShotsOTFor', 'ShotsOTAgainst', 'CornersFor', 'CornersAgainst', 'FoulsFor', 'FoulsAgainst', 'YellowsFor', 'YellowsAgainst', 'RedsFor', 'RedsAgainst']
for col in stat_columns:
    team_form[f'Overall_Last5_{col}']=team_form.groupby('Team')[col].transform(lambda x: x.shift(1).rolling(5,min_periods=1).mean())

cols_to_keep = ['Date', 'Team'] + [f'Overall_Last5_{col}' for col in stat_columns]

team_form=team_form[cols_to_keep]

In [17]:
df = df.merge(team_form, left_on=['Date','HomeTeam'], right_on=['Date','Team'], how='left')
df.rename(columns=lambda x: f"Home_{x}" if x in team_form.columns and x not in ['Date', 'Team'] else x, inplace=True)
df.drop('Team',axis=1,inplace=True)

df = df.merge(team_form, left_on=['Date','AwayTeam'], right_on=['Date','Team'], how='left')
df.rename(columns=lambda x: f"Away_{x}" if x in team_form.columns and x not in ['Date', 'Team'] else x, inplace=True)
df.drop('Team',axis=1,inplace=True)


In [18]:
kolumny_do_podgladu = [
    'Date', 'HomeTeam', 'AwayTeam',
    'Home_Last5_HomePts', 'Away_Last5_AwayPts',
    'Home_Overall_Last5_Points', 'Away_Overall_Last5_Points',
    'Home_Overall_Last5_ShotsOTFor',
    'Away_Overall_Last5_YellowsFor'
]
print(df[kolumny_do_podgladu].tail(10))

           Date         HomeTeam         AwayTeam  Home_Last5_HomePts  \
9709 2026-03-15  Nott'm Forest_0         Fulham_3                 0.6   
9710 2026-03-16      Brentford_0         Wolves_2                 1.0   
9711 2026-03-20    Bournemouth_1     Man United_0                 1.8   
9712 2026-03-21        Everton_0        Chelsea_0                 1.0   
9713 2026-03-21       Brighton_0      Liverpool_0                 1.0   
9714 2026-03-21         Fulham_3        Burnley_4                 1.8   
9715 2026-03-21          Leeds_2      Brentford_0                 1.2   
9716 2026-03-22    Aston Villa_1       West Ham_2                 0.8   
9717 2026-03-22      Newcastle_2     Sunderland_3                 1.2   
9718 2026-03-22      Tottenham_0  Nott'm Forest_0                 0.2   

      Away_Last5_AwayPts  Home_Overall_Last5_Points  \
9709                 0.8                        0.4   
9710                 0.6                        1.6   
9711                 1.6       

# Head-to-Head (H2H) Statistics
I'm calculating H2H statistics for each matchup. This includes the average points, goals for, and goals against from the last 3 encounters between the two teams at the home team's venue, as well as the overall H2H record from their last 5 meetings.

In [19]:
df = df.sort_values(by='Date').reset_index(drop=True)
df['H2H_Home_Pts']=df.groupby(['HomeTeam', 'AwayTeam'])['HomePoints'].transform(lambda x: x.shift(1).rolling(3,min_periods=1).mean())
df['H2H_Home_GF']=df.groupby(['HomeTeam', 'AwayTeam'])['FTHG'].transform(lambda x: x.shift(1).rolling(3,min_periods=1).mean())
df['H2H_Home_GA']=df.groupby(['HomeTeam', 'AwayTeam'])['FTAG'].transform(lambda x: x.shift(1).rolling(3,min_periods=1).mean())



In [20]:
h2h_home = df[['Date','HomeTeam', 'AwayTeam', 'HomePoints','FTHG','FTAG']].copy()
h2h_home.columns = ['Date', 'Team', 'Opponent', 'Points', 'GF', 'GA']

h2h_away = df[['Date', 'AwayTeam', 'HomeTeam', 'AwayPoints','FTAG','FTHG']].copy()
h2h_away.columns = ['Date', 'Team', 'Opponent', 'Points','GF', 'GA']

h2h_all = pd.concat([h2h_home, h2h_away]).sort_values(by=['Team', 'Opponent','Date']).reset_index(drop=True)
h2h_all['H2H_Overall_Pts']= h2h_all.groupby(['Team','Opponent'])['Points'].transform(lambda x: x.shift(1).rolling(5,min_periods=1).mean())
h2h_all['H2H_Overall_GF']=h2h_all.groupby(['Team','Opponent'])['GF'].transform(lambda x: x.shift(1).rolling(5,min_periods=1).mean())
h2h_all['H2H_Overall_GA']=h2h_all.groupby(['Team','Opponent'])['GA'].transform(lambda x: x.shift(1).rolling(5,min_periods=1).mean())

h2h_all = h2h_all[['Date','Team','Opponent','H2H_Overall_Pts','H2H_Overall_GF','H2H_Overall_GA']]



In [21]:
df = df.merge(h2h_all, left_on=['Date', 'HomeTeam','AwayTeam'], right_on=['Date','Team','Opponent'], how='left')
df.rename(columns={
    'H2H_Overall_Pts': 'Home_H2H_Overall_Pts',
    'H2H_Overall_GF': 'Home_H2H_Overall_GF',
    'H2H_Overall_GA': 'Home_H2H_Overall_GA',
}, inplace=True)
df.drop(['Team','Opponent'], axis=1, inplace=True)

df = df.merge(h2h_all, left_on=['Date', 'AwayTeam', 'HomeTeam'], right_on=['Date','Team','Opponent'], how='left')
df.rename(columns={
    'H2H_Overall_Pts': 'Away_H2H_Overall_Pts',
    'H2H_Overall_GF': 'Away_H2H_Overall_GF',
    'H2H_Overall_GA': 'Away_H2H_Overall_GA',
}, inplace=True)

df.drop(['Team','Opponent'], axis=1, inplace=True)


In [22]:
print(df[['Date','HomeTeam','AwayTeam','Home_DaysOfRest','Away_DaysOfRest']].tail(10))

           Date       HomeTeam         AwayTeam  Home_DaysOfRest  \
9709 2026-03-15    Liverpool_0      Tottenham_0             12.0   
9710 2026-03-16    Brentford_0         Wolves_2             13.0   
9711 2026-03-20  Bournemouth_1     Man United_0              6.0   
9712 2026-03-21     Brighton_0      Liverpool_0              7.0   
9713 2026-03-21       Fulham_3        Burnley_4              6.0   
9714 2026-03-21      Everton_0        Chelsea_0              7.0   
9715 2026-03-21        Leeds_2      Brentford_0              6.0   
9716 2026-03-22    Newcastle_2     Sunderland_3              8.0   
9717 2026-03-22  Aston Villa_1       West Ham_2              7.0   
9718 2026-03-22    Tottenham_0  Nott'm Forest_0              7.0   

      Away_DaysOfRest  
9709             10.0  
9710             13.0  
9711              5.0  
9712              6.0  
9713              7.0  
9714              7.0  
9715              5.0  
9716              8.0  
9717              8.0  
9718       

# ELO Rating System

I'm implementing an ELO rating system to quantify the relative strength of each team. The ELO rating of a team is updated after each match based on the match outcome and the expected outcome. I'm also incorporating a home-field advantage (HFA) and a goal difference modifier to make the ratings more accurate.

In [23]:
elo_dict={}
K_FACTOR = 20
HFA =70
SCALE_FACTOR = 400

def expected_result(elo_a, elo_b):
    return 1 / (1+10**((elo_b-elo_a)/ SCALE_FACTOR))

def get_g_modifier(goal_diff):
    if goal_diff <=1:
        return 1.0
    elif goal_diff == 2:
        return 1.5
    else:
        return (11 + goal_diff) / 8.0

In [24]:
home_elos_before = []
away_elos_before = []

for index, row in df.iterrows():
    home = row['HomeTeam']
    away = row['AwayTeam']

    if home not in elo_dict:
        elo_dict[home] = 1500 if row['Season']=='2000/2001' else 1350
    if away not in elo_dict:
        elo_dict[away] = 1500 if row['Season']=='2000/2001' else 1350

    elo_h = elo_dict[home]
    elo_a = elo_dict[away]
    home_elos_before.append(elo_h)
    away_elos_before.append(elo_a)

    elo_h_adj = elo_h + HFA

    exp_h= expected_result(elo_h_adj, elo_a)
    exp_a= expected_result(elo_a, elo_h_adj)

    goals_h = row['FTHG']
    goals_a = row['FTAG']
    goals_diff = abs(goals_h - goals_a)

    G = get_g_modifier(goals_diff)

    if goals_h > goals_a:
        res_h, res_a = 1.0, 0.0
    elif goals_h == goals_a:
        res_h, res_a = 0.5, 0.5
    else:
        res_h, res_a = 0.0, 1.0

    elo_dict[home] = elo_h + K_FACTOR * G * (res_h - exp_h)
    elo_dict[away] = elo_a + K_FACTOR * G * (res_a - exp_a)

df['Home_ELO'] = home_elos_before
df['Away_ELO'] = away_elos_before

In [25]:
print(df[['Date','HomeTeam','AwayTeam','FTHG','FTAG','Home_ELO','Away_ELO']].tail(10))

           Date       HomeTeam         AwayTeam  FTHG  FTAG     Home_ELO  \
9709 2026-03-15    Liverpool_0      Tottenham_0   1.0   1.0  1664.863133   
9710 2026-03-16    Brentford_0         Wolves_2   2.0   2.0  1551.763603   
9711 2026-03-20  Bournemouth_1     Man United_0   2.0   2.0  1537.762575   
9712 2026-03-21     Brighton_0      Liverpool_0   2.0   1.0  1557.306957   
9713 2026-03-21       Fulham_3        Burnley_4   3.0   1.0  1507.330348   
9714 2026-03-21      Everton_0        Chelsea_0   3.0   0.0  1524.313738   
9715 2026-03-21        Leeds_2      Brentford_0   0.0   0.0  1438.433108   
9716 2026-03-22    Newcastle_2     Sunderland_3   1.0   2.0  1562.817728   
9717 2026-03-22  Aston Villa_1       West Ham_2   2.0   0.0  1579.005987   
9718 2026-03-22    Tottenham_0  Nott'm Forest_0   0.0   3.0  1438.133965   

         Away_ELO  
9709  1431.098417  
9710  1384.346271  
9711  1587.762308  
9712  1657.827585  
9713  1332.028024  
9714  1620.655654  
9715  1545.826725  
971

In [26]:
df['HomeTeam'] = df['HomeTeam'].str.split('_').str[0]
df['AwayTeam'] = df['AwayTeam'].str.split('_').str[0]

In [27]:
print(df[df['HomeTeam'] == 'Liverpool'][['Date', 'HomeTeam', 'AwayTeam', 'H2H_Home_Pts', 'Home_H2H_Overall_Pts']].tail(10))


           Date   HomeTeam       AwayTeam  H2H_Home_Pts  Home_H2H_Overall_Pts
9526 2025-11-22  Liverpool  Nott'm Forest      2.000000                   2.0
9545 2025-12-03  Liverpool     Sunderland           NaN                   NaN
9561 2025-12-13  Liverpool       Brighton      2.333333                   1.4
9585 2025-12-27  Liverpool         Wolves      3.000000                   3.0
9599 2026-01-01  Liverpool          Leeds           NaN                   1.0
9625 2026-01-17  Liverpool        Burnley           NaN                   3.0
9644 2026-01-31  Liverpool      Newcastle      3.000000                   2.6
9658 2026-02-08  Liverpool       Man City      2.333333                   1.6
9684 2026-02-28  Liverpool       West Ham      3.000000                   2.6
9709 2026-03-15  Liverpool      Tottenham      3.000000                   2.4


In [28]:
df['ELO_difference']=(df['Home_ELO']+70) - df['Away_ELO']


In [29]:
import json

elo_dict_clean = {}
for key,value in elo_dict.items():
    clean_key = key.split('_')[0]
    elo_dict_clean[clean_key] = value
with open("../data/Premier_League/elo_ratings.json", 'w') as f:
    json.dump(elo_dict_clean, f)
print("ELO SAVED!")
print(f"Arsenal: {elo_dict_clean.get('Arsenal'):.1f}")
print(f"Chelsea: {elo_dict_clean.get('Chelsea'):.1f}")

ELO SAVED!
Arsenal: 1770.0
Chelsea: 1601.8


# Final Data Cleaning and Saving
In this final step, I'm cleaning the data to prepare it for machine learning. This involves:
- Filling missing values for form and H2H features with reasonable defaults.
- Dropping columns that represent the actual match stats (since they would be "leaky" features for prediction).
- Saving the final, cleaned DataFrame to a new CSV file.

In [30]:
safe_historical_data = df[df['Date'].dt.year <= 2002]

cols_form = [col for col in df.columns if 'Last5' in col]

for col in cols_form:
    low_baseline = safe_historical_data[col].quantile(0.25)

    if pd.isna(low_baseline):
        low_baseline = 0.0
    df[col] = df[col].fillna(low_baseline)

cols_h2h_pts = [col for col in df.columns if 'H2H' in col and 'Pts' in col]
cols_h2h_goals =  [col for col in df.columns if 'H2H' in col and ('GF' in col or 'GA' in col)]

for col in cols_h2h_pts:
    df[col] = df[col].fillna(1.0)

for col in cols_h2h_goals:
    df[col] = df[col].fillna(0.0)



In [31]:
current_match_stats = [
    'FTHG', 'FTAG', 'HTHG', 'HTAG', 'HTR', 'FTR',
    'HS', 'AS', 'HST', 'AST',
    'HF', 'AF', 'HC', 'AC',
    'HY', 'AY', 'HR', 'AR',
    'HomePoints', 'AwayPoints'
]

cols_to_drop = [col for col in current_match_stats if col in df.columns]
df = df.drop(columns=cols_to_drop)

In [32]:
print(df.isnull().sum()[df.isnull().sum() > 0])

Series([], dtype: int64)


In [33]:
save_path = "../data/Premier_League/PremierLeague_Match_Data_Ready_For_ML.csv"
df.to_csv(save_path, index=False)